# Bank Marketing Response Model

**Purpose:** evaluate whether a bank can rank prospective marketing contacts before a call using only pre-call information, while making the validation boundary visible.

**Dataset:** UCI Bank Marketing `bank-additional-full.csv`, a public dataset from Portuguese direct-marketing campaigns from 2008 to 2010.

**Method:** download and checksum the public ZIP, validate schema, convert `unknown` values to missing values, exclude `duration` as post-call leakage, freeze a 75% source-order outer cutoff, choose the model recipe inside the early segment with four expanding-window folds, refit on all early rows, and score the late segment once.

**Metric:** average precision is primary because positive responses are uncommon. ROC AUC, calibration bands, cumulative gains, and fixed contact-budget lift curves support the readout.

**Headline takeaway:** the model selected by mean fold average precision is not meaningfully superior: fold prevalence and average precision vary sharply, and pooled early out-of-fold ranking is weak. The late top 10% budget underperforms the late base rate. The late `1.69x` result is an exploratory top-1% concentration, not policy evidence without economics, stability analysis, and a further untouched evaluation.


## Imports And Configuration

The notebook uses reusable project helpers for data integrity, leakage-safe preprocessing, source-order validation, ranking metrics, and shared figure styling.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.pipeline import Pipeline

from ml_portfolio.bank_marketing import (
    BANK_DATA_SHA256,
    BANK_DATA_URL,
    LEAKY_FEATURE,
    RANDOM_STATE,
    build_bank_pipeline,
    load_bank_marketing_data,
    make_bank_preprocessor,
    prepare_bank_modeling_data,
    run_forward_bank_validation,
)
from ml_portfolio.plotting import (
    ACCENT,
    CAPTION,
    HIGHLIGHT,
    MUTED,
    apply_portfolio_style,
    save_figure,
)
from ml_portfolio.ranking import (
    calibration_by_score_band,
    cumulative_gains_frame,
    expanding_window_splits,
    fixed_budget_table,
    source_order_split_indices,
)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "assets").exists() and (PROJECT_ROOT.parent / "assets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

apply_portfolio_style()
pd.options.display.float_format = "{:.4f}".format

OUTER_TRAIN_FRACTION = 0.75
INNER_N_SPLITS = 4


## Load And Verify The Public CSV

The source file is not committed to the repository. It is downloaded from UCI at runtime, checked for minimum size, and verified against the expected SHA-256 digest before the nested CSV is read.


In [ ]:
bank_raw = load_bank_marketing_data()

print(f"Rows: {bank_raw.shape[0]:,} | Columns: {bank_raw.shape[1]}")
print(f"Source URL: {BANK_DATA_URL}")
print(f"Expected ZIP SHA-256: {BANK_DATA_SHA256}")
display(bank_raw.head())


## Target Balance And Missing-Like Values

The target is whether a client subscribed to a term deposit. The positive class is uncommon, and several categorical fields use `unknown` as a missing-like value rather than a clean category.


In [ ]:
target_balance = bank_raw["y"].value_counts(normalize=True).rename("share")
unknown_counts = (bank_raw == "unknown").sum().sort_values(ascending=False)
unknown_counts = unknown_counts[unknown_counts > 0].rename("unknown_count")

display(target_balance.to_frame())
display(unknown_counts.to_frame())


## Define The Leakage Boundary

`duration` records how long the call lasted. It is only known after the contact happens, so it is excluded from every pre-call model. The feature appears later only in a controlled leakage demonstration.


In [ ]:
X, y, feature_roles = prepare_bank_modeling_data(bank_raw)

role_rows = []
for feature in feature_roles["numeric_features"]:
    role_rows.append({"feature": feature, "role": "numeric pre-call"})
for feature in feature_roles["categorical_features"]:
    role_rows.append({"feature": feature, "role": "categorical pre-call"})
role_rows.append({"feature": LEAKY_FEATURE, "role": "excluded post-call leakage"})

feature_role_table = pd.DataFrame(role_rows)
display(feature_role_table)
print(f"Positive rate: {y.mean():.3f}")
print(f"Excluded leakage feature: {LEAKY_FEATURE}")


## Freeze The Source-Order Validation Design

The outer split is frozen before model selection: the earliest 75% of source-order rows form the development segment, and the latest 25% are held back for one stress-test evaluation. Because the public CSV has row order but no complete row-level timestamp, this is an order-based temporal stress test, not a formal timestamped deployment validation.

Inside the early segment, four expanding-window folds are used. The first fold trains on the first half of the early segment and validates on the next contiguous block; each later fold expands the train window and validates on the next block. This keeps all model family and hyperparameter choices inside the early period.


In [ ]:
early_indices, late_indices = source_order_split_indices(
    len(X),
    train_fraction=OUTER_TRAIN_FRACTION,
)
inner_min_train_size = len(early_indices) // 2
inner_splits = list(
    expanding_window_splits(
        len(early_indices),
        n_splits=INNER_N_SPLITS,
        min_train_size=inner_min_train_size,
    )
)

outer_split_table = pd.DataFrame(
    [
        {
            "segment": "early development",
            "start_row": int(early_indices[0]),
            "end_row_inclusive": int(early_indices[-1]),
            "rows": len(early_indices),
            "positive_rate": y.iloc[early_indices].mean(),
        },
        {
            "segment": "late stress test",
            "start_row": int(late_indices[0]),
            "end_row_inclusive": int(late_indices[-1]),
            "rows": len(late_indices),
            "positive_rate": y.iloc[late_indices].mean(),
        },
    ]
)

fold_rows = []
for fold, (train_index, validation_index) in enumerate(inner_splits, start=1):
    fold_rows.append(
        {
            "fold": fold,
            "train_start": int(train_index[0]),
            "train_end_inclusive": int(train_index[-1]),
            "validation_start": int(validation_index[0]),
            "validation_end_inclusive": int(validation_index[-1]),
            "train_rows": len(train_index),
            "validation_rows": len(validation_index),
            "validation_positive_rate": y.iloc[early_indices[validation_index]].mean(),
        }
    )

inner_fold_table = pd.DataFrame(fold_rows)
display(outer_split_table)
display(inner_fold_table)


## Select The Model Recipe Inside The Early Segment

Candidate recipes are deliberately small: a majority-class dummy model, balanced logistic regression at three regularization strengths, and class-weighted random forests with two leaf-size and feature-subset settings. Average precision is the selection metric because the practical problem is ranking rare responders.


In [ ]:
result = run_forward_bank_validation(
    X,
    y,
    numeric_features=feature_roles["numeric_features"],
    categorical_features=feature_roles["categorical_features"],
    train_fraction=OUTER_TRAIN_FRACTION,
    n_splits=INNER_N_SPLITS,
    min_train_size=inner_min_train_size,
)

cv_results = result.selection.cv_results.copy()
fold_results = result.selection.fold_results.copy()
selected_fold_results = fold_results.loc[
    fold_results["model"] == result.selection.selected_name
].reset_index(drop=True)
display(cv_results)
display(fold_results)
print(f"Selected recipe: {result.selection.selected_name}")
print(f"Selected params: {result.selection.selected_params}")


## Early Segment Diagnostics

The predeclared selection rule remains mean fold average precision, using early rows only. The fold table retains each candidate's validation prevalence, average precision, AP lift over that fold's base rate, and ROC AUC. For the selected forest, fold AP ranges from `0.046` to `0.266` as prevalence ranges from `0.055` to `0.176`; AP lift ranges from `0.83x` to `1.51x`. Its mean-fold AP of `0.108` versus dummy AP of `0.087` is unstable and should not be read as meaningful superiority.

The pooled early out-of-fold readout is a separate diagnostic across all validation rows: AP `0.078` against prevalence `0.087`, with ROC AUC `0.470`. These early diagnostics are not used to choose a business-value threshold, and no late row influences model or policy selection.


In [ ]:
early_diagnostic_metrics = {
    "validation_design": "Early expanding-window diagnostics",
    "model": result.selection.selected_name,
    "rows_with_oof_scores": len(result.selection.early_oof_labels),
    "base_positive_rate": result.selection.early_oof_base_positive_rate,
    "average_precision": result.selection.early_oof_average_precision,
    "average_precision_lift_over_base_rate": (
        result.selection.early_oof_average_precision
        / result.selection.early_oof_base_positive_rate
    ),
    "roc_auc": result.selection.early_oof_roc_auc,
}
early_budget_table = fixed_budget_table(
    result.selection.early_oof_labels,
    result.selection.early_oof_scores,
)

display(selected_fold_results)
display(pd.DataFrame([early_diagnostic_metrics]))
display(early_budget_table)


## One Late-Segment Evaluation

After selection, the chosen recipe is refit on all early rows and scored once on the untouched late segment. The fixed-budget readout asks what happens if a reviewer predeclares a contact capacity, such as the top 10% of scored records. This is a policy sensitivity view, not a fabricated profit model.


In [ ]:
late_metrics = result.late_metrics.copy()
late_budget_table = result.late_budget_table.copy()
late_top_10 = late_budget_table.loc[late_budget_table["budget_share"] == 0.10].iloc[0]
late_top_1 = late_budget_table.loc[late_budget_table["budget_share"] == 0.01].iloc[0]
late_metrics.update(
    {
        "top_1_percent_lift": late_top_1["lift_vs_base"],
        "top_10_percent_lift": late_top_10["lift_vs_base"],
        "top_10_percent_response_rate": late_top_10["response_rate"],
    }
)

display(pd.DataFrame([late_metrics]))
display(late_budget_table)


## Precision-Recall Curve

The late stress-test curve is compared with the late base response rate. If the curve sits close to or below the base line for much of the range, the model is not a reliable broad targeting rule under this source-order split.


In [ ]:
early_precision, early_recall, _ = precision_recall_curve(
    result.selection.early_oof_labels,
    result.selection.early_oof_scores,
)
late_precision, late_recall, _ = precision_recall_curve(result.late_labels, result.late_scores)

fig, ax = plt.subplots()
ax.plot(
    early_recall,
    early_precision,
    color=MUTED,
    linewidth=1.8,
    label=(
        "Early pooled OOF "
        f"AP={early_diagnostic_metrics['average_precision']:.3f}"
    ),
)
ax.plot(
    late_recall,
    late_precision,
    color=ACCENT,
    linewidth=2.2,
    label=f"Late stress test AP={late_metrics['average_precision']:.3f}",
)
ax.axhline(
    late_metrics["base_positive_rate"],
    color=HIGHLIGHT,
    linestyle="--",
    linewidth=1.2,
    label=f"Late base rate={late_metrics['base_positive_rate']:.3f}",
)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Late-period precision-recall is weak under source-order validation")
ax.legend(loc="upper right")
ax.text(
    0,
    -0.22,
    "UCI Bank Marketing; duration excluded; late rows not used for selection.",
    transform=ax.transAxes,
    color=CAPTION,
    fontsize=9,
)
fig.tight_layout()
save_figure(fig, "bank_marketing_precision_recall", project_root=PROJECT_ROOT)
plt.show()


## Fixed Contact-Budget Curve

A fixed budget curve is easier to defend than a threshold optimized for a single score. The default policy readout uses top 10%, while 1%, 5%, 20%, and 30% show sensitivity to capacity.


In [ ]:
fig, ax = plt.subplots()
for table, label, color in [
    (early_budget_table, "Early pooled OOF", MUTED),
    (late_budget_table, "Late stress test", ACCENT),
]:
    ax.plot(
        table["budget_share"] * 100,
        table["lift_vs_base"],
        marker="o",
        linewidth=2.0,
        color=color,
        label=label,
    )
ax.axhline(1.0, color=HIGHLIGHT, linestyle="--", linewidth=1.2, label="Base rate")
ax.set_xlabel("Contact budget (% of scored records)")
ax.set_ylabel("Lift versus segment base rate")
ax.set_title("Exploratory top-1% late concentration\ndoes not persist at wider budgets")
ax.set_xticks([1, 5, 10, 20, 30])
ax.legend(loc="upper right")
ax.text(
    0,
    -0.22,
    "Not policy evidence: no economics, stability analysis, or further untouched evaluation.",
    transform=ax.transAxes,
    color=CAPTION,
    fontsize=9,
)
fig.tight_layout()
save_figure(fig, "bank_marketing_contact_budget_curve", project_root=PROJECT_ROOT)
plt.show()


## Cumulative Gains

The gains curve shows how quickly responders are captured as the contact budget expands. The late curve is weaker than the early diagnostics, which is the central validation finding.


In [ ]:
early_gains = cumulative_gains_frame(
    result.selection.early_oof_labels,
    result.selection.early_oof_scores,
    "Early pooled OOF",
)
late_gains = cumulative_gains_frame(result.late_labels, result.late_scores, "Late stress test")

fig, ax = plt.subplots()
for frame, color in [(early_gains, MUTED), (late_gains, ACCENT)]:
    ax.plot(
        frame["share_contacted"],
        frame["responders_captured_share"],
        linewidth=2.2,
        color=color,
        label=frame["validation_design"].iloc[0],
    )
ax.plot([0, 1], [0, 1], color=HIGHLIGHT, linestyle="--", linewidth=1.2, label="Random ranking")
ax.axvline(0.10, color="#A9B1BC", linestyle=":", linewidth=1.2)
ax.set_xlim(0, 1)
ax.set_xlabel("Share of contacts scored")
ax.set_ylabel("Share of responders captured")
ax.set_title("Cumulative gains under source-order validation", pad=16)
ax.set_ylim(0, 1.03)
ax.legend(loc="lower right")
ax.text(
    0,
    -0.22,
    (
        "Top-10% lift: early "
        f"{early_budget_table.loc[early_budget_table['budget_share'] == 0.10, 'lift_vs_base'].iloc[0]:.2f}x; "
        f"late {late_top_10['lift_vs_base']:.2f}x."
    ),
    transform=ax.transAxes,
    color=CAPTION,
    fontsize=9,
)
fig.tight_layout()
save_figure(fig, "bank_marketing_cumulative_gains", project_root=PROJECT_ROOT)
plt.show()


## Calibration Diagnostic

Calibration is treated as a diagnostic here, not as an automatically added transformation. The late bands show that the selected model's probability scale is poorly calibrated under the source-order stress test.


In [ ]:
late_calibration = calibration_by_score_band(result.late_labels, result.late_scores, n_bins=10)
display(late_calibration)

fig, ax = plt.subplots()
ax.plot([0, 1], [0, 1], color=MUTED, linestyle="--", linewidth=1.2, label="Perfect calibration")
ax.scatter(
    late_calibration["mean_score"],
    late_calibration["response_rate"],
    color=ACCENT,
    s=64,
    label="Late score bands",
)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Mean predicted response score")
ax.set_ylabel("Observed response rate")
ax.set_title("Late-period scores are not calibrated probabilities")
ax.legend(loc="upper left")
ax.text(
    0,
    -0.22,
    "Calibration is reported as a diagnostic only; no calibration model is fitted here.",
    transform=ax.transAxes,
    color=CAPTION,
    fontsize=9,
)
fig.tight_layout()
save_figure(fig, "bank_marketing_calibration", project_root=PROJECT_ROOT)
plt.show()


## Leakage Demonstration: Why `duration` Is Excluded

This diagnostic stays inside the early segment. It shows how much a post-call feature can inflate validation scores when the task is supposed to be pre-call ranking.


In [ ]:
comparison_frame = X.copy()
comparison_frame[LEAKY_FEATURE] = bank_raw[LEAKY_FEATURE]
leakage_rows = []
for include_duration in [False, True]:
    numeric_features = list(feature_roles["numeric_features"])
    comparison_features = list(X.columns)
    if include_duration:
        numeric_features = [*numeric_features, LEAKY_FEATURE]
        comparison_features = [*comparison_features, LEAKY_FEATURE]

    fold_scores = []
    for train_index, validation_index in inner_splits:
        pipeline = Pipeline(
            steps=[
                (
                    "preprocess",
                    make_bank_preprocessor(
                        numeric_features=numeric_features,
                        categorical_features=feature_roles["categorical_features"],
                    ),
                ),
                (
                    "model",
                    LogisticRegression(
                        max_iter=1000,
                        class_weight="balanced",
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        )
        early_train_index = early_indices[train_index]
        early_validation_index = early_indices[validation_index]
        pipeline.fit(comparison_frame.iloc[early_train_index][comparison_features], y.iloc[early_train_index])
        probability = pipeline.predict_proba(
            comparison_frame.iloc[early_validation_index][comparison_features]
        )[:, 1]
        fold_scores.append(average_precision_score(y.iloc[early_validation_index], probability))

    leakage_rows.append(
        {
            "feature_set": "with duration (leaky)" if include_duration else "without duration",
            "early_fold_average_precision_mean": np.mean(fold_scores),
            "early_fold_average_precision_std": np.std(fold_scores),
        }
    )

leakage_comparison = pd.DataFrame(leakage_rows)
display(leakage_comparison)


## Late-Segment Permutation Importance

Permutation importance is calculated on the untouched late segment after the final model is fit on early rows. It is interpretive evidence, not a new selection step.


In [ ]:
importance = permutation_importance(
    result.fitted_model,
    X.iloc[result.outer_split.late_indices],
    y.iloc[result.outer_split.late_indices],
    scoring="average_precision",
    n_repeats=5,
    random_state=RANDOM_STATE,
)
importance_table = pd.DataFrame(
    {
        "feature": X.columns,
        "importance_mean": importance.importances_mean,
        "importance_std": importance.importances_std,
    }
).sort_values("importance_mean", ascending=False)
display(importance_table.head(12))

top_importance = importance_table.head(10).sort_values("importance_mean")
fig, ax = plt.subplots()
colors = [ACCENT if value > 0 else MUTED for value in top_importance["importance_mean"]]
bars = ax.barh(top_importance["feature"], top_importance["importance_mean"], color=colors)
ax.bar_label(bars, labels=[f"{value:.3f}" for value in top_importance["importance_mean"]], padding=3, fontsize=9)
ax.set_xlabel("Mean decrease in late average precision after permutation")
ax.set_ylabel("Feature")
ax.set_title("Late-period importance estimates are descriptive and small")
ax.text(
    0,
    -0.22,
    "Computed after final fit; duration remains excluded from the pre-call model.",
    transform=ax.transAxes,
    color=CAPTION,
    fontsize=9,
)
fig.tight_layout()
save_figure(fig, "bank_marketing_permutation_importance", project_root=PROJECT_ROOT)
plt.show()


## Error Review By Contact Channel

The late-period review groups broad errors by contact channel. This is not a fairness audit. It is a small model-behavior check to keep the notebook from stopping at aggregate metrics.


In [ ]:
late_review = X.iloc[result.outer_split.late_indices].copy()
late_review["actual"] = result.late_labels
late_review["score"] = result.late_scores
late_review["in_top_10_percent"] = False
top_10_count = int(late_top_10["contacts"])
top_10_index = late_review.sort_values("score", ascending=False).head(top_10_count).index
late_review.loc[top_10_index, "in_top_10_percent"] = True

channel_review = (
    late_review.groupby(["contact", "in_top_10_percent"], dropna=False)
    .agg(
        rows=("actual", "size"),
        response_rate=("actual", "mean"),
        mean_score=("score", "mean"),
    )
    .reset_index()
    .sort_values(["contact", "in_top_10_percent"])
)
display(channel_review)


## Conclusion

The workflow verifies the public source, handles missing-like values, excludes `duration` as post-call leakage, keeps model selection inside the early source-order segment, and evaluates the late segment once.

The selected random forest ranks first under the predeclared mean-fold AP rule, but the fold prevalence and AP ranges make that comparison unstable; it should not be read as meaningful superiority. Pooled early OOF AP is `0.078` against prevalence `0.087`, with ROC AUC `0.470`. On the late segment, average precision is below the late base response rate, ROC AUC is below `0.50`, and the top 10% fixed budget has lift below `1.0x`. The `1.69x` value is an exploratory top-1% concentration. It is not policy evidence without campaign economics, stability analysis, and a further untouched evaluation.
